In [1]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler, StandardScaler

df = pd.read_csv('spotify_artist_streaming_2020_2025.csv')
df.head()

,track_id,track_name,artist_name,album_name,release_date,genre,duration_ms,popularity,danceability,energy,...,popularity_category,loudness_category,key_name,mode_name,is_explicit_bool,release_quarter,is_weekend_release,log_stream_count,upbeat_score,artist_track_count
0,346d85e2-ffde-41b3-854a-589ed880b91f,Storm,Ethereal,The Hollow Journeys,2021-10-06,Pop,169594,47,0.6777,0.7949,...,Medium,Loud,A,Major,False,Q4,False,10.7492,0.6632,3232
1,12bc268a-363b-4f19-8fa8-0ced7ae1e462,Silent,The Wolfs,Ancient Thunder,2024-10-26,K-Pop,256755,26,0.9405,0.7163,...,Medium,Moderate,C#,Major,False,Q4,True,8.5895,0.7615,20
2,30d627d9-878d-4bb3-be59-71feb6ed4f3f,Shattered Lines,LolaOre,Twisted Harbors,2022-11-12,Punk,170212,46,0.6853,0.9500,...,Medium,Loud,E,Major,False,Q4,True,12.2935,0.7828,199
3,85501bb9-d221-4339-9e2e-29b1b3daff0a,Breaking Roses,Clay Black,The Blinding Letters,2020-06-22,Pop,181250,4,0.7817,0.6943,...,Low,Moderate,F#,Major,False,Q2,False,8.5658,0.6778,693
4,7bf92bd2-c6fe-4869-a90e-242bb8bbe7a3,Lines,Knot Hart,The Cosmic Letters,2024-01-22,Latin,160894,27,0.7200,0.4958,...,Medium,Moderate,C#,Major,False,Q1,False,10.2191,0.5494,12111


In [3]:
print("Количество пропусков до обработки:")
print(df.isnull().sum())

Количество пропусков до обработки:
track_id               0
track_name             0
artist_name            0
album_name             0
release_date           0
genre                  0
duration_ms            0
popularity             0
danceability           0
energy                 0
key                    0
loudness               0
mode                   0
instrumentalness       0
tempo                  0
stream_count           0
country                0
explicit               0
label                  0
release_year           0
release_month          0
release_day_of_week    0
duration_minutes       0
popularity_category    0
loudness_category      0
key_name               0
mode_name              0
is_explicit_bool       0
release_quarter        0
is_weekend_release     0
log_stream_count       0
upbeat_score           0
artist_track_count     0
dtype: int64


In [4]:
print("Пропуски до манипуляции (должны быть нули):")
print(df[['stream_count', 'duration_ms', 'genre']].isnull().sum())

df.loc[0, 'stream_count'] = None
df.loc[0, 'genre'] = None

print("\nПропуски ПОСЛЕ искусственного создания:")
print(df[['stream_count', 'genre']].isnull().sum())

df['stream_count'] = df['stream_count'].fillna(df['stream_count'].median())

df['genre'] = df['genre'].fillna(df['genre'].mode()[0])

print("\nПропуски ПОСЛЕ заполнения (снова должны быть нули):")
print(df[['stream_count', 'genre']].isnull().sum())

print("\nПервая строка после заполнения:")
print(df[['stream_count', 'genre']].head(1))

Пропуски до манипуляции (должны быть нули):
stream_count    0
duration_ms     0
genre           0
dtype: int64

Пропуски ПОСЛЕ искусственного создания:
stream_count    1
genre           1
dtype: int64

Пропуски ПОСЛЕ заполнения (снова должны быть нули):
stream_count    0
genre           0
dtype: int64

Первая строка после заполнения:
   stream_count genre
0       15954.0   R&B


In [12]:
minmax_scaler = MinMaxScaler()
std_scaler = StandardScaler()

df['stream_count_MinMax'] = minmax_scaler.fit_transform(df[['stream_count']])

df['duration_ms_Standard'] = std_scaler.fit_transform(df[['duration_ms']])

df[['stream_count', 'stream_count_MinMax', 'duration_ms', 'duration_ms_Standard']].head()

,stream_count,stream_count_MinMax,duration_ms,duration_ms_Standard
0,46592,0.001073,169594,-0.900883
1,5374,0.000122,256755,1.051404
2,218262,0.005033,170212,-0.887041
3,5248,0.000119,181250,-0.639805
4,27420,0.000630,160894,-1.095752


In [2]:
# Удаляем столбцы, которые не несут полезной информации для модели
columns_to_drop = ['track_id', 'track_name', 'artist_name', 'album_name', 'release_date']
df = df.drop(columns=columns_to_drop, errors='ignore')

print("Столбцы после удаления:")
print(df.columns.tolist())

# Автоматически находим все текстовые столбцы
text_columns = df.select_dtypes(include=['object']).columns.tolist()
print(f"Найдены текстовые столбцы: {text_columns}")

# Применяем One-Hot Encoding
if text_columns:
    df = pd.get_dummies(df, columns=text_columns, drop_first=True)
    print(f"После кодирования в датафрейме {df.shape[1]} столбцов.")
else:
    print("Текстовых столбцов не найдено.")

# Проверяем, что не осталось текстовых столбцов
remaining_text = df.select_dtypes(include=['object']).columns.tolist()
print(f"Осталось текстовых столбцов: {remaining_text}")

Столбцы после удаления:
['genre', 'duration_ms', 'popularity', 'danceability', 'energy', 'key', 'loudness', 'mode', 'instrumentalness', 'tempo', 'stream_count', 'country', 'explicit', 'label', 'release_year', 'release_month', 'release_day_of_week', 'duration_minutes', 'popularity_category', 'loudness_category', 'key_name', 'mode_name', 'is_explicit_bool', 'release_quarter', 'is_weekend_release', 'log_stream_count', 'upbeat_score', 'artist_track_count']
Найдены текстовые столбцы: ['genre', 'country', 'label', 'release_day_of_week', 'popularity_category', 'loudness_category', 'key_name', 'mode_name', 'release_quarter']
После кодирования в датафрейме 122 столбцов.
Осталось текстовых столбцов: []


In [3]:
df.to_csv('processed_spotify_data.csv', index=False)

print("Файл 'processed_spotify_data.csv' успешно сохранен в папку с проектом")

Файл 'processed_spotify_data.csv' успешно сохранен в папку с проектом
